# Chapter 5 — Classifying and making a decision · Annotated solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch05_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer starts with the reasoning on paper, then shows its equivalent in Python. For each calculation, the comments point out the group that serves as the denominator.

**Contents.** Data · The calculations of lessons 5.1 to 5.6 (likelihood, softmax) · Exercises 5.1 to 5.6

**Notebook conventions.**
- Decision rule (equation 5.1): a message is flagged if its estimated probability is **greater than or equal to** the threshold. For the twelve messages, the rule is applied to the **unrounded** value `p_exact`, recomputed with the sigmoid; the book's table only shows it rounded to two decimals. In the exercises where the probabilities are given as decimals, those decimals are the values to use.
- A measure whose denominator is zero (for example the precision when no alert is raised) is `None` and is displayed as “undefined”.
- The F1 is computed as $2\,TP / (2\,TP + FP + FN)$ (equation 5.8).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colors of the book: one color = one role
SPAM, LEGIT, FLAGGED, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
THRESHOLD, ORANGE, BLUE, GREEN = '#8e24aa', '#eb6834', '#2a78d6', '#2e8b57'   # threshold; sigmoid; loss; filter B


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def show_value(v):
    """Notebook convention: a measure whose denominator is zero is None and is displayed as “undefined”."""
    return 'undefined' if v is None else f'{v:.2f}'

## Data

The identifiers D5.1, D5.2, D5.E1, D5.E2 and D5.E3 are those of the book and of the CSV files.

In [ ]:
# Chapter 5 data, copied from the book's CSV files (do not modify)

# D5.1 — twelve messages: true label, links, suspicious words, estimated probability ROUNDED to two decimals
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4, 0.95),
    ('M2', 'legitime', 2, 0, 0.12),
    ('M3', 'legitime', 4, 0, 0.50),
    ('M4', 'spam', 3, 3, 0.62),
    ('M5', 'legitime', 1, 0, 0.05),
    ('M6', 'legitime', 2, 2, 0.27),
    ('M7', 'spam', 2, 3, 0.38),
    ('M8', 'legitime', 2, 1, 0.18),
    ('M9', 'spam', 4, 4, 0.88),
    ('M10', 'legitime', 1, 1, 0.08),
    ('M11', 'legitime', 0, 0, 0.02),
    ('M12', 'legitime', 0, 1, 0.03)
], columns=['message', 'label', 'links', 'suspicious_words', 'score'])

# D5.2 — filter A: 1,000 messages by bin (count, spams, mean predicted probability)
calibration = pd.DataFrame([
    (0.0, 0.1, 540, 22, 0.03),
    (0.1, 0.3, 150, 29, 0.19),
    (0.3, 0.5, 80, 33, 0.40),
    (0.5, 0.7, 60, 35, 0.60),
    (0.7, 0.9, 90, 71, 0.80),
    (0.9, 1.0, 80, 77, 0.95)
], columns=['bin_min', 'bin_max', 'messages', 'spams', 'mean_proba'])

# D5.E1 — filter B: same table (exercise 5.1)
calibration_b = pd.DataFrame([
    (0.0, 0.1, 400, 40, 0.02),
    (0.1, 0.3, 175, 42, 0.18),
    (0.3, 0.5, 100, 43, 0.41),
    (0.5, 0.7, 100, 55, 0.61),
    (0.7, 0.9, 150, 93, 0.82),
    (0.9, 1.0, 75, 56, 0.97)
], columns=['bin_min', 'bin_max', 'messages', 'spams', 'mean_proba'])

# D5.E2 — ten messages from the association (exercise 5.2): these decimals are the values to use
ten = pd.DataFrame([
    ('N1', 'spam', 0.94),
    ('N2', 'legitime', 0.07),
    ('N3', 'legitime', 0.66),
    ('N4', 'spam', 0.50),
    ('N5', 'legitime', 0.30),
    ('N6', 'spam', 0.21),
    ('N7', 'legitime', 0.42),
    ('N8', 'spam', 0.77),
    ('N9', 'legitime', 0.13),
    ('N10', 'spam', 0.58)
], columns=['message', 'label', 'score'])

# D5.E3 — four messages scored by two filters (exercise 5.3)
four = pd.DataFrame([
    ('P1', 'spam', 0.80, 0.99),
    ('P2', 'spam', 0.60, 0.90),
    ('P3', 'legitime', 0.40, 0.10),
    ('P4', 'legitime', 0.20, 0.01)
], columns=['message', 'label', 'p_filter_A', 'p_filter_B'])

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `message` | message identifier (M1…) | — | — |
| `label` | true label: spam or legitimate | — | `etiquette` |
| `links` | number of links in the message | links | `liens` |
| `suspicious_words` | number of suspicious words in the message | words | `mots_suspects` |
| `score` | estimated spam probability, rounded to two decimals | — | — |
| `bin_min` | lower bound of the probability bin | — | `tranche_min` |
| `bin_max` | upper bound of the probability bin | — | `tranche_max` |
| `messages` | number of messages | messages | — |
| `spams` | number of spam messages | messages | — |
| `mean_proba` | mean predicted probability in the bin | — | `proba_moyenne` |
| `p_filter_A` | spam probability given by filter A | — | `p_filtre_A` |
| `p_filter_B` | spam probability given by filter B | — | `p_filtre_B` |
| `threshold` | decision threshold | — | `seuil` |
| `twelve` | the twelve messages of chapter 5 | — | — |

## The calculations of the lessons, in Python

These cells redo the calculations of lessons 5.1 to 5.6, including the likelihood (end of lesson 5.4) and the softmax (lesson 5.6), with their short questions.

In [ ]:
# Exact values of the twelve messages (equations 5.3 and 5.4): the rule is applied to these
w, b = np.array([1, 0.5]), -4
X = twelve[['links', 'suspicious_words']].to_numpy()
z = X @ w + b
p_exact = sigmoid(z)
is_spam = (twelve['label'] == 'spam').to_numpy()
y = is_spam.astype(int)                 # 1 = spam, 0 = legitimate

# Lesson 5.1 — moving the threshold (equations 5.1 and 5.2)
for threshold in (0.5, 0.25):
    flagged = p_exact >= threshold                    # rule: greater than OR EQUAL, on the exact value
    assert (flagged == (twelve['score'] >= threshold)).all()   # here, rounding changes no decision
    tp, fn = int((flagged & is_spam).sum()), int((~flagged & is_spam).sum())
    fp, tn = int((flagged & ~is_spam).sum()), int((~flagged & ~is_spam).sum())
    print(f'threshold {threshold}: TP = {tp}, FN = {fn}, FP = {fp}, TN = {tn} | '
          f'recall = {tp}/{tp + fn} = {tp / (tp + fn):.3f}, '            # denominator: the spam row
          f'false alarms = {fp}/{fp + tn} = {fp / (fp + tn):.3f}, '   # denominator: the legitimate row
          f'precision = {tp}/{tp + fp} = {tp / (tp + fp):.3f}')         # denominator: the flagged column
switched = twelve['message'][(p_exact >= 0.25) & (p_exact < 0.5)]
print('messages that switch from 0.5 to 0.25:', list(switched))

fig, axes = plt.subplots(2, 1, figsize=(7, 3.4), sharex=True)
for ax, threshold in zip(axes, (0.5, 0.25)):
    for q, s in zip(p_exact, is_spam):
        sig = q >= threshold
        ax.scatter(q, 1 if s else 0, s=70, marker='s' if s else 'o', color=SPAM if s else LEGIT,
                   edgecolor=FLAGGED if sig else 'gray', linewidth=2.4 if sig else 0.6)
    ax.axvline(threshold, color=THRESHOLD, linewidth=2.5)
    ax.set_yticks([0, 1], ['legitimate', 'spams']); ax.set_ylim(-0.6, 1.6)
    ax.set_title(f'threshold {threshold}', loc='left', color=THRESHOLD)
axes[1].set_xlabel('Estimated probability (exact value)'); plt.tight_layout(); plt.show()

In [ ]:
# Lesson 5.2 — raw score, estimated probability, decision; the case of M3; calibration
table = pd.DataFrame({'message': twelve['message'], 'z': z, 'p_exact': p_exact.round(4),
                        'rounded p': p_exact.round(2),
                        'decision (threshold 0.5)': np.where(p_exact >= 0.5, 'flagged', 'no')})
print(table.to_string(index=False))
print('rounded = scores of lesson 5.1:', np.allclose(p_exact.round(2), twelve['score']))
print(f"M3: z = {z[2]}, e^0 = {np.exp(0.0)}, p̂ = 1/(1 + 1) = {p_exact[2]} exactly → "
      f"{'flagged' if p_exact[2] >= 0.5 else 'not flagged'} at threshold 0.5")
print(f'M4: e^(-0.5) = {np.exp(-0.5):.4f}, sigmoid(0.5) = {sigmoid(0.5):.3f}; M7: sigmoid(-0.5) = {sigmoid(-0.5):.3f}')
print('threshold 0.5 on p̂ ⇔ threshold 0 on z:', bool(((p_exact >= 0.5) == (z >= 0)).all()),
      f'| threshold 0.25 on p̂ ⇔ z ≥ {np.log(0.25 / 0.75):.2f}')

zs = np.linspace(-5, 4.5, 300)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoid(zs), color=ORANGE, linewidth=2.2)
plt.axhline(0.5, color=THRESHOLD, linewidth=2)
plt.scatter(z[is_spam], p_exact[is_spam], marker='s', color=SPAM, zorder=3, label='spams')
plt.scatter(z[~is_spam], p_exact[~is_spam], marker='o', color=LEGIT, edgecolor='gray', zorder=3, label='legitimate')
plt.xlabel('Raw score z'); plt.ylabel('Estimated probability p̂'); plt.legend(); plt.grid(alpha=0.4); plt.show()

# An estimated probability can be checked: side by side, mean predicted p̂, observed frequency, count (denominator)
cal = calibration.copy()
cal['observed frequency'] = (cal['spams'] / cal['messages']).round(2)
print(cal[['bin_min', 'bin_max', 'mean_proba', 'observed frequency', 'messages', 'spams']].to_string(index=False))
t = cal.iloc[4]
print(f"bin 0.7–0.9: predicted {t['mean_proba']:.2f}, observed {int(t['spams'])} / {int(t['messages'])} ≈ "
      f"{t['spams'] / t['messages']:.2f} | expected {t['mean_proba'] * t['messages']:.0f} spams, "
      f"standard deviation {np.sqrt(t['messages'] * 0.8 * 0.2):.1f}")
print('largest gap between predicted and observed:', f"{np.max(np.abs(cal['mean_proba'] - cal['spams'] / cal['messages'])):.3f}")

In [ ]:
# Lesson 5.3 — the penalty for a true spam: −ln(p̂) (equation 5.5). np.log is the NATURAL logarithm.
for p in (0.90, 0.50, 0.10, 0.01):
    print(f'p̂ = {p:.2f} → penalty = {-np.log(p):.3f}')
print(f'ratios: {np.log(0.01) / np.log(0.5):.1f} times the 0.50 version, '
      f'{np.log(0.01) / np.log(0.9):.1f} times the 0.90 version')
p_spams = p_exact[is_spam]                             # UNROUNDED values from lesson 5.2 (M4: 0.622)
penalties = -np.log(p_spams)
print('spams', list(twelve.loc[is_spam, 'message']), ': penalties', penalties.round(2), f'| mean = {penalties.mean():.2f}')
print(f'product of the probabilities = {p_spams.prod():.3f}; −ln(product) = {-np.log(p_spams.prod()):.2f} '
      f'= sum of the penalties = {penalties.sum():.2f}')
ps = np.linspace(0.006, 1, 300)
plt.figure(figsize=(6, 3))
plt.plot(ps, -np.log(ps), color=BLUE, linewidth=2.2)
plt.scatter([0.9, 0.5, 0.01], -np.log([0.9, 0.5, 0.01]), marker='s', color=SPAM, zorder=3)
plt.xlabel('Estimated probability p̂ for a true spam'); plt.ylabel('Penalty −ln(p̂)'); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lesson 5.4 — the loss of each message (equation 5.6) and the mean loss (equation 5.7)
def loss(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))   # −ln of the probability given to the right answer

p_tab = twelve['score'].to_numpy()                       # values shown in the table (to two decimals)
losses = loss(y, p_exact)                              # losses computed on the UNROUNDED values, as in the book
print(pd.DataFrame({'message': twelve['message'], 'y': y, 'p̂': p_tab,
                    'correct answer': np.where(y == 1, p_tab, 1 - p_tab), 'loss': losses.round(2)}).to_string(index=False))
print(f'M4: −ln({p_exact[3]:.3f}) = {losses[3]:.3f} ≈ {losses[3]:.2f} (with the rounded 0.62, we would get {-np.log(0.62):.2f})')
print(f'M3: −ln(1 − 0.50) = {-np.log(0.5):.3f}; M6: −ln(1 − 0.27) = {-np.log(0.73):.3f}; M11: {-np.log(0.98):.3f}')
print(f'sum of the rounded losses = {losses.round(2).sum():.2f}; mean loss = {losses.mean():.2f} '
      f'(unrounded sum: {losses.sum():.3f})')

# Fitting first: the mean loss as a function of b (w fixed), like the loss curve of chapter 3
def loss_b(bb):
    return loss(y, sigmoid(X @ w + bb)).mean()
for bb in (-5, -4, -3):
    print(f'b = {bb}: mean loss = {loss_b(bb):.2f}')
bs = np.linspace(-7, -1, 6001)
Lb = [loss_b(bb) for bb in bs]
print(f'bottom of the curve: b ≈ {bs[int(np.argmin(Lb))]:.2f}, loss {min(Lb):.4f} (versus {loss_b(-4):.4f} for b = −4)')
plt.figure(figsize=(6, 3))
plt.plot(bs, Lb, color=BLUE, linewidth=2.2)
plt.scatter([-4], [loss_b(-4)], marker='D', color=ORANGE, zorder=3)
plt.xlabel('Constant b'); plt.ylabel('Mean loss'); plt.grid(alpha=0.4); plt.show()
print('Squared error for the spam scored 0.01:', round((1 - 0.01) ** 2, 2), '; for 0.50:', (1 - 0.5) ** 2)

In [ ]:
# Lesson 5.4 (continued) — why a logarithm? The likelihood of the twelve labels
# q = probability the filter gives to the class ACTUALLY observed: p̂ for a spam, 1 − p̂ for a legitimate message
q = np.where(y == 1, p_exact, 1 - p_exact)
V = q.prod()                                   # messages independent given the model: we multiply
print('q:', q.round(3))
print(f'likelihood V = {V:.4f}')
print(f'−ln V = {-np.log(V):.3f} = sum of the twelve losses {losses.sum():.3f}; '
      f'divided by 12: {-np.log(V) / 12:.3f} = mean loss')
print(f'A thousand messages of this quality: V ≈ 10^{1000 / 12 * np.log10(V):.1f} (too small without care)')
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
axes[0].bar(twelve['message'], q, color=np.where(y == 1, SPAM, LEGIT))
axes[0].set_ylabel('q (observed class)'); axes[0].set_ylim(0, 1)
axes[1].barh(0, -np.log(q), left=np.concatenate([[0], np.cumsum(-np.log(q))[:-1]]),
             color=np.where(y == 1, SPAM, LEGIT), edgecolor='white')
axes[1].set_yticks([]); axes[1].set_xlabel('−ln q placed end to end = −ln V')
plt.tight_layout(); plt.show()

In [ ]:
# Short question (lesson 5.4) — likelihood of the eight legitimate messages
V_legit = q[y == 0].prod()   # 0.881 × 0.500 × 0.953 × 0.731 × 0.818 × 0.924 × 0.982 × 0.971

In [ ]:
# Check of the short question
if V_legit is None:
    print('not yet completed. Complete the line.')
else:
    V_spams = q[y == 1].prod()
    print(f'spams: {V_spams:.3f}; legitimate: {V_legit:.3f}; product: {V_spams * V_legit:.4f} (V = {V:.4f}) →',
          'correct' if abs(V_legit - q[y == 0].prod()) < 1e-9 else 'check again')
    print(f'in logarithms: {-np.log(V_spams):.3f} + {-np.log(V_legit):.3f} = {-np.log(V_spams) - np.log(V_legit):.3f}')

In [ ]:
# Lesson 5.5 — comparing thresholds: alerts, false alarms, missed, precision, recall, F1, cost (k = 3)
rows = []
for threshold in (0.25, 0.3, 0.5, 0.6, 0.7, 0.96):
    sig = p_exact >= threshold                              # exact values
    tp, fp = int((sig & is_spam).sum()), int((sig & ~is_spam).sum())
    fn = int((~sig & is_spam).sum())
    precision = None if tp + fp == 0 else tp / (tp + fp)   # flagged column empty → undefined
    rows.append({'threshold': threshold, 'alerts': tp + fp, 'false alarms': fp, 'missed': fn,
                   'precision': show_value(precision), 'recall': f'{tp / (tp + fn):.2f}',
                   'F1': f'{2 * tp / (2 * tp + fp + fn):.2f}', 'cost (k = 3)': fn + 3 * fp})
print(pd.DataFrame(rows).to_string(index=False))
print(f'F1 at threshold 0.3: 2 × 0.8 × 1 / 1.8 = {2 * 0.8 * 1 / 1.8:.2f}; at threshold 0.6: 1.5 / 1.75 = {1.5 / 1.75:.2f}')

# The same calculation with scikit-learn, if it is installed: its convention for an undefined precision is 0
try:
    from sklearn.metrics import precision_score, f1_score
    predicted_label = (p_exact >= 0.96).astype(int)
    print('scikit-learn at threshold 0.96: precision =', precision_score(y, predicted_label, zero_division=0),
          '(convention zero_division=0); F1 =', f1_score(y, predicted_label, zero_division=0))
except ImportError:
    print('scikit-learn not installed: this calculation is optional.')

## Lesson 5.6 — More than two classes: the softmax

Three mutually exclusive folders: spam, legitimate promotion, ordinary message. The softmax (equation 5.10) turns three scores into three probabilities that add up to 1; with two classes, it gives back the sigmoid (equation 5.11).

In [ ]:
# Lesson 5.6 — the softmax (equation 5.10)
def softmax(scores):
    e = np.exp(np.asarray(scores, dtype=float))   # the exponential makes each score positive, keeping the order
    return e / e.sum()                            # dividing by the sum gives probabilities that add up to 1

folders = ['spam', 'legitimate promotion', 'ordinary message']
scores_M3 = [2, 1, 0]                             # newsletter M3; observed class: legitimate promotion
p3 = softmax(scores_M3)
print(pd.DataFrame({'folder': folders, 'z': scores_M3, 'e^z': np.exp(scores_M3).round(3), 'probability': p3.round(3)}).to_string(index=False))
print(f'sum of the exponentials = {np.exp(scores_M3).sum():.3f}; sum of the probabilities = {p3.sum():.3f}')
print(f'loss of M3 = −ln {p3[1]:.3f} = {-np.log(p3[1]):.3f}')
p_better = softmax([0, 2, 0])
print(f'better-tuned scores (0, 2, 0): promotion {p_better[1]:.3f}, loss {-np.log(p_better[1]):.3f}')
# Two classes (equation 5.11): legitimate at score 0, spam at raw score z
print(f'M7 (z = −0.5): two-class softmax {softmax([-0.5, 0])[0]:.3f} = sigmoid {sigmoid(-0.5):.3f}')
plt.figure(figsize=(6, 2.6))
plt.bar(folders, p3, color=[SPAM, BLUE, LEGIT], edgecolor=['none', 'black', 'none'], linewidth=2)
plt.ylabel('probability'); plt.ylim(0, 1); plt.title('softmax of the scores (2, 1, 0) — observed class outlined', loc='left')
plt.show()

In [ ]:
# Short questions (lesson 5.6)
q1_probas = softmax([1, 1, 1])                                   # each one is 1/3
q2_same = bool(np.allclose(softmax([3, 2, 1]), softmax([2, 1, 0])))   # +1 everywhere: nothing changes
q3_p_M4 = softmax([0.5, 0])[0]                                   # = sigmoid(0.5 − 0) (equation 5.11)

In [ ]:
# Check of the short questions
if q1_probas is None or q2_same is None or q3_p_M4 is None:
    print('not yet completed. Complete the three lines.')
else:
    print('1.', np.round(q1_probas, 3), f'; loss = ln 3 = {np.log(3):.3f} →',
          'correct' if np.allclose(q1_probas, 1 / 3) else 'check again')
    print('2.', q2_same, '→', 'correct' if q2_same is True else 'check again')
    print(f'3. {q3_p_M4:.3f} →', 'correct' if abs(q3_p_M4 - sigmoid(0.5)) < 1e-9 else 'check again')

## Exercise 5.1 — Reading a graph: two filters against the diagonal

*Concepts: estimated probability and calibration (lesson 5.2).* First read the figure in the book. Filter A: data D5.2; filter B: data D5.E1.

### a) to d) Annotated solution

Here, the denominator of each frequency is **the count of the bin**, not the full set of 1,000 messages.

**a)** For filter A, in the bin where it predicts 0.80 on average, the observed frequency is about 0.79: this is the highlighted row of the table in lesson 5.2 (71 spams out of 90 messages).

**b)** Filter B predicts 0.97 on average in its last bin, but only about 0.75 of spams are observed there. In its first bin, it predicts 0.02 and we observe 0.10.

**c)** The diagonal is perfect calibration: observed frequency = predicted probability. The points of A follow it to within 0.02. Those of B move away from it at both ends: at the top, B announces more spams than there are (0.82 → 0.62; 0.97 → 0.75); at the bottom, it announces fewer (0.02 → 0.10; 0.18 → 0.24). B is **overconfident**: its probabilities are too close to 0 and to 1.

**d)** Yes: the observed frequency of B rises from one bin to the next (0.10, 0.24, 0.43, 0.55, 0.62, 0.75). Its **ranking** of the messages therefore remains useful, and we can still choose a threshold with a confusion matrix. But its “0.97” cannot be read as 97 chances out of 100: among the messages scored that way, we observe about 3 spams out of 4.

In [ ]:
freq_b = (calibration_b['spams'] / calibration_b['messages']).to_numpy()   # denominator: the bin
expected_last = calibration_b['mean_proba'].iloc[-1] * calibration_b['messages'].iloc[-1]   # 0.97 × 75

In [ ]:
# Check of e)
if freq_b is None or expected_last is None:
    print('not yet completed. Complete the previous cell.')
else:
    print('frequencies of B:', np.round(freq_b, 2), '→',
          'correct' if np.allclose(freq_b, [0.10, 0.24, 0.43, 0.55, 0.62, 56 / 75]) else 'check again')
    n, q = calibration_b['messages'].iloc[-1], calibration_b['mean_proba'].iloc[-1]
    sd = np.sqrt(n * q * (1 - q))                       # standard deviation of the number of spams (exercise 4.6)
    observed_freq = calibration_b['spams'].iloc[-1]
    print(f'bin 0.9–1: expected {expected_last:.2f} spams, observed {observed_freq}, standard deviation {sd:.2f}, '
          f'gap = {(expected_last - observed_freq) / sd:.0f} standard deviations →',
          'correct' if abs(expected_last - 72.75) < 0.01 else 'check again')
    plt.figure(figsize=(4.2, 4.2))
    plt.plot([0, 1], [0, 1], ':', color='gray', label='perfect calibration')
    plt.plot(calibration['mean_proba'], calibration['spams'] / calibration['messages'], 'o-', color=BLUE, label='filter A')
    plt.plot(calibration_b['mean_proba'], freq_b, '^--', color=GREEN, label='filter B')
    plt.xlabel('Mean predicted probability'); plt.ylabel('Observed frequency'); plt.legend(); plt.grid(alpha=0.4)
    plt.show()

### e) Annotated solution

Let us take the reference model of lesson 5.2: independent labels, predicted probabilities assumed to be correct (here 0.97 for each message in the bin). We would expect $0.97 \times 75 \approx 72.75$ spams. We observe 56. In this model, the standard deviation of the number of spams is $\sqrt{75 \times 0.97 \times 0.03} \approx 1.48$ (exercise 4.6). The gap, about 17 spams, represents more than ten standard deviations: this is not the chance of a small sample. 56 spams are not compatible with this model, and filter B really is overconfident. For filter A, the same comparison in the bin 0.7–0.9 gave 72 expected for 71 observed, with a standard deviation of 3.8: a result compatible with the reference model.

## Exercise 5.2 — Three thresholds for an association

*Concepts: decision rule and confusion matrix (lesson 5.1), recall, false-alarm rate and precision (equation 5.2), F1 and cost (lesson 5.5).* Ten messages checked by the association's office (data D5.E2), three thresholds: 0.3, 0.5 and 0.7. **The probabilities in the table are the values to use as they are.** Do questions a) to f) on paper first.

### a) to c) Annotated solution

**a)** On the axis, from left to right: N2 (0.07, legitimate), N9 (0.13, legitimate), N6 (0.21, spam), N5 (0.30, legitimate), N7 (0.42, legitimate), N4 (0.50, spam), N10 (0.58, spam), N3 (0.66, legitimate), N8 (0.77, spam), N1 (0.94, spam). Two messages fall exactly on a threshold: N5 on 0.3 and N4 on 0.5. Since these decimals are the values supplied by the filter, there is no hidden rounding to worry about.

**b)** The “greater than or equal” rule flags N4 at threshold 0.5 and N5 at threshold 0.3. The matrices (rows: what the message is; columns: flagged, not flagged):

| Threshold | TP (spams flagged) | FN (spams missed) | FP (false alarms) | TN (legitimate let through) |
|:--|--:|--:|--:|--:|
| 0.7 | 2 (N1, N8) | 3 (N10, N4, N6) | 0 | 5 |
| 0.5 | 4 (N1, N8, N10, N4) | 1 (N6) | 1 (N3) | 4 |
| 0.3 | 4 | 1 (N6) | 3 (N3, N7, N5) | 2 |

Check: in each matrix, $TP + FN = 5$ spams and $FP + TN = 5$ legitimate messages.

**c)** Recall is counted **in the spam row** (5 messages), the false-alarm rate **in the legitimate-message row** (5 messages), precision **in the flagged column**, whose size changes with the threshold.

| Threshold | Recall | False-alarm rate | Precision |
|:--|--:|--:|--:|
| 0.7 | 2/5 = 0.4 | 0/5 = 0 | 2/2 = 1 |
| 0.5 | 4/5 = 0.8 | 1/5 = 0.2 | 4/5 = 0.8 |
| 0.3 | 4/5 = 0.8 | 3/5 = 0.6 | 4/7 ≈ 0.57 |

At threshold 0.95, no message reaches the threshold (the highest is 0.94): the flagged column is empty, its count is 0, and precision is **undefined**. It is neither 0 nor 1. The notebook displays it as “undefined”; scikit-learn would replace it with 0, with a warning. Recall, for its part, is 0.

### b) The confusion matrix in Python

In [ ]:
def confusion_matrix(scores, is_spam, threshold):
    """scores and is_spam: two lists of the same length. Returns the dictionary
    {'TP': ..., 'FN': ..., 'FP': ..., 'TN': ...}. Rule: flagged if score >= threshold."""
    m = {'TP': 0, 'FN': 0, 'FP': 0, 'TN': 0}
    for s, spam in zip(scores, is_spam):
        flagged = s >= threshold                    # greater than OR EQUAL: N4 and N5 count
        if spam:
            m['TP' if flagged else 'FN'] += 1   # spam row
        else:
            m['FP' if flagged else 'TN'] += 1   # legitimate-message row
    return m

In [ ]:
# Check: first on the twelve messages (EXACT values), then on the ten messages of the exercise
scenario = [('lesson, threshold 0.5', list(p_exact), list(is_spam), 0.5, (3, 1, 1, 7)),
       ('lesson, threshold 0.25', list(p_exact), list(is_spam), 0.25, (4, 0, 2, 6)),
       ('exercise, threshold 0.7', list(ten['score']), list(ten['label'] == 'spam'), 0.7, (2, 3, 0, 5)),
       ('exercise, threshold 0.5', list(ten['score']), list(ten['label'] == 'spam'), 0.5, (4, 1, 1, 4)),
       ('exercise, threshold 0.3', list(ten['score']), list(ten['label'] == 'spam'), 0.3, (4, 1, 3, 2))]
for name, sc, sp, threshold, expected in scenario:
    m = confusion_matrix(sc, sp, threshold)
    if m is None:
        print(f'{name}: not yet completed. Complete the function.')
    else:
        obtained = (m['TP'], m['FN'], m['FP'], m['TN'])
        print(f'{name}: TP, FN, FP, TN = {obtained} →', 'correct' if obtained == expected else 'check again')

### c) and e) The measures and the F1

In [ ]:
def metrics(m):
    def quotient(num, den):
        return None if den == 0 else num / den   # empty group: the measure is undefined
    recall = quotient(m['TP'], m['TP'] + m['FN'])                 # spam row
    false_alarm_rate = quotient(m['FP'], m['FP'] + m['TN'])   # legitimate-message row
    precision = quotient(m['TP'], m['TP'] + m['FP'])              # flagged column
    f1 = quotient(2 * m['TP'], 2 * m['TP'] + m['FP'] + m['FN'])   # defined even without any alert
    return recall, false_alarm_rate, precision, f1

In [ ]:
# Check of c) and e)
scores_e2, spams_e2 = list(ten['score']), list(ten['label'] == 'spam')
if confusion_matrix(scores_e2, spams_e2, 0.5) is None:
    print('Complete confusion_matrix first.')
elif metrics(confusion_matrix(scores_e2, spams_e2, 0.5))[0] is None:
    print('Measures: not yet completed. Complete the function metrics.')
else:
    expected_values = {0.7: (0.4, 0.0, 1.0), 0.5: (0.8, 0.2, 0.8), 0.3: (0.8, 0.6, 4 / 7)}
    for threshold, att in expected_values.items():
        r, f, p, _ = metrics(confusion_matrix(scores_e2, spams_e2, threshold))
        print(f'threshold {threshold}: recall = {show_value(r)}, false alarms = {show_value(f)}, precision = {show_value(p)} →',
              'correct' if np.allclose((r, f, p), att) else 'check again')
    r, f, p, f1 = metrics(confusion_matrix(scores_e2, spams_e2, 0.95))
    print(f'threshold 0.95: no alert; recall = {show_value(r)}, precision = {show_value(p)}, F1 = {show_value(f1)} →',
          'correct' if r == 0 and p is None and f1 == 0 else 'check again')
    f1s = [metrics(confusion_matrix(scores_e2, spams_e2, s))[3] for s in (0.7, 0.5, 0.3)]
    print('F1 at thresholds 0.7 / 0.5 / 0.3:', [round(v, 2) for v in f1s], '→',
          'correct' if np.allclose(f1s, [4 / 7, 0.8, 8 / 12]) else 'check again')

### f) The cost of errors

In [ ]:
def cost(m, k):
    return m['FN'] + k * m['FP']   # one false alarm is “worth” k missed spams

In [ ]:
# Check of f)
if confusion_matrix(scores_e2, spams_e2, 0.5) is None or cost({'FN': 0, 'FP': 0}, 3) is None:
    print('Cost: not yet completed. Complete the functions.')
else:
    for k, expected in ((3, [3, 4, 10]), (1, [3, 2, 4])):
        costs = [cost(confusion_matrix(scores_e2, spams_e2, s), k) for s in (0.7, 0.5, 0.3)]
        best_one = (0.7, 0.5, 0.3)[costs.index(min(costs))]
        print(f'k = {k}: costs at thresholds 0.7 / 0.5 / 0.3 = {costs}, best threshold {best_one} →',
              'correct' if costs == expected else 'check again')

### g) All values of the threshold

In [ ]:
# Recall, precision and F1 for the thresholds 0, 0.01, …, 1 (undefined precision: no point)
if confusion_matrix(scores_e2, spams_e2, 0.5) is None or metrics(confusion_matrix(scores_e2, spams_e2, 0.5))[0] is None:
    print('Complete confusion_matrix and metrics to plot the curves.')
else:
    thresholds = np.round(np.arange(0, 1.001, 0.01), 2)
    R, F, P, F1 = zip(*[metrics(confusion_matrix(scores_e2, spams_e2, s)) for s in thresholds])
    P = [np.nan if p is None else p for p in P]
    plt.figure(figsize=(6, 3.2))
    plt.step(thresholds, R, where='post', color=SPAM, linewidth=2, label='recall (spam row)')
    plt.step(thresholds, P, where='post', color=FLAGGED, linewidth=2, linestyle='--', label='precision (flagged column)')
    plt.step(thresholds, F1, where='post', color='black', linewidth=1.5, linestyle=':', label='F1')
    for s in (0.3, 0.5, 0.7):
        plt.axvline(s, color=THRESHOLD, linewidth=1, alpha=0.6)
    plt.xlabel('Threshold'); plt.ylim(-0.05, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### d) to g) Annotated solution

**d)** From 0.5 to 0.3, two messages switch: N7 (0.42) and N5 (0.30, flagged because the rule says “greater than **or equal**”). Both are **legitimate** messages. We gain nothing: no extra spam is caught, recall stays at 0.8, because the only missed spam, N6 (0.21), is still below the threshold. We lose two more false alarms: the rate goes from 0.2 to 0.6 and precision from 0.8 to 4/7 ≈ 0.57.

**e)** With $F_1 = 2\,TP / (2\,TP + FP + FN)$: threshold 0.7: $4 / (4 + 0 + 3) = 4/7 \approx 0.57$; threshold 0.5: $8 / (8 + 1 + 1) = 0.8$; threshold 0.3: $8 / (8 + 3 + 1) = 8/12 \approx 0.67$. The F1 points to threshold **0.5**. At threshold 0.95, with no alert, precision is undefined but the F1 is $0 / (0 + 0 + 5) = 0$.

**f)** With a false alarm that costs three missed spams: threshold 0.7: $3 + 3 \times 0 = 3$; threshold 0.5: $1 + 3 \times 1 = 4$; threshold 0.3: $1 + 3 \times 3 = 10$. Threshold **0.7** is the least costly: it lets three spams through, but loses none of the association's messages. It is not the F1 threshold. In $F_1 = 2\,TP / (2\,TP + FP + FN)$, $FP$ and $FN$ appear symmetrically, with the same coefficient. From 0.7 to 0.5, we catch two more spams (N10 and N4) at the price of one false alarm (N3): $FP + FN$ goes from 3 to 2 and $TP$ from 2 to 4, so the F1 goes up. The office's rule gives three cost units to a false alarm and one to each missed spam: the cost changes by $-2 + 3 = +1$. If both errors cost the same ($k = 1$), the costs become 3, 2 and 4, and 0.5 wins, as with the F1. The choice of threshold depends on what each error costs, and it is up to the office to say. Finally, check the whole picture: these measures are based on **ten messages**, a sample far too small to set a final threshold.

**g)** As the threshold rises, recall can only go down, step by step, each time a spam leaves the flagged column. Precision has no fixed direction of change, because its denominator changes at each step. Beyond 0.94, the precision curve stops: it is undefined, since there are no alerts any more. The F1, for its part, drops to 0.

## Exercise 5.3 — Interpreting: same decisions, different losses

*Concepts: log loss (lessons 5.3 and 5.4), decision rule (equation 5.1).* Data D5.E3: these probabilities are the values to use.

### a) to d) Annotated solution

**a)** At threshold 0.5, both filters flag P1 and P2 and let P3 and P4 through: no error, neither for A nor for B. Their confusion matrices are identical.

**b)** We take $-\ln$ of the probability given to the right answer: $\hat p$ for a spam, $1 - \hat p$ for a legitimate message.

- Filter A: $-\ln(0.8) \approx 0.223$; $-\ln(0.6) \approx 0.511$; $-\ln(1 - 0.4) = -\ln(0.6) \approx 0.511$; $-\ln(1 - 0.2) = -\ln(0.8) \approx 0.223$. Mean: $1.468 / 4 \approx 0.37$.
- Filter B: $-\ln(0.99) \approx 0.010$; $-\ln(0.9) \approx 0.105$; $-\ln(0.9) \approx 0.105$; $-\ln(0.99) \approx 0.010$. Mean: $0.231 / 4 \approx 0.06$.

**c)** B is preferable: it makes the same decisions, but with more confidence, and rightly so. The confusion matrix only looks at which side of the threshold each probability falls on; the loss looks at how far it is from the right answer. That is why we fit the weights with the loss, not with the number of errors.

**d)** C differs from B only on P2, scored 0.01: $-\ln(0.01) \approx 4.605$. Its mean loss is $(0.010 + 4.605 + 0.105 + 0.010) / 4 \approx 1.18$, more than three times that of A. Yet at threshold 0.5, C makes only one error (it misses P2). A single error made with confidence is enough to make the loss explode: this is intended (lesson 5.3).

In [ ]:
def mean_loss(labels, probas):
    """labels: list of 1 (spam) and 0 (legitimate); probas: estimated probabilities of being spam."""
    y_, p_ = np.array(labels), np.array(probas, dtype=float)
    return float(np.mean(-(y_ * np.log(p_) + (1 - y_) * np.log(1 - p_))))   # mean over the messages

In [ ]:
# Check
y4 = list((four['label'] == 'spam').astype(int))
if mean_loss(y4, four['p_filter_A']) is None:
    print('not yet completed. Complete the function.')
else:
    p_c = [0.99, 0.01, 0.10, 0.01]
    for name, probas, att in (('A', four['p_filter_A'], 0.3670), ('B', four['p_filter_B'], 0.0577), ('C', p_c, 1.1827)):
        v = mean_loss(y4, probas)
        print(f'filter {name}: mean loss = {v:.4f} →', 'correct' if abs(v - att) < 0.0001 else 'check again')
    print('check with the twelve messages of lesson 5.4:', round(mean_loss(list(y), list(p_exact)), 2))

## Exercise 5.4 — Cost decides the threshold

*Concepts: cost of a threshold (equation 5.9), F1 (equation 5.8).* We return to the twelve messages, with their **exact** probabilities `p_exact`: the rule is applied to these.

### a) Expected prediction

The best threshold **goes up**: when false alarms are expensive, we accept letting spams through in order to raise fewer alarms.

In [ ]:
def total_cost(threshold, k):
    flagged = p_exact >= threshold                    # EXACT values: M6 is 0.2689…, not 0.27
    missed = int((~flagged & is_spam).sum())    # spam row
    false_alarms = int((flagged & ~is_spam).sum())    # legitimate-message row
    return missed + k * false_alarms

In [ ]:
# b) to e) The best thresholds for each k, then the F1 thresholds
thresholds = np.round(np.arange(0, 1.001, 0.01), 2)
if total_cost(0.5, 3) is None:
    print('not yet completed. Complete the function.')
else:
    expected_values = {0.1: (0.27, 0.37), 1: (0.27, 0.62), 3: (0.51, 0.62), 10: (0.51, 0.62)}
    plt.figure(figsize=(6, 3.2))
    for k, att in expected_values.items():
        c = np.array([total_cost(s, k) for s in thresholds])
        best_ones = thresholds[np.isclose(c, c.min())]
        print(f'k = {k:>4}: minimum cost {c.min():.1f}, reached for thresholds from {best_ones.min():.2f} to {best_ones.max():.2f} →',
              'correct' if (best_ones.min(), best_ones.max()) == att else 'check again')
        plt.step(thresholds, c / c.max(), where='post', label=f'k = {k}')
    plt.xlabel('Threshold'); plt.ylabel('Cost (divided by its maximum)'); plt.legend(fontsize=8); plt.grid(alpha=0.4)
    plt.show()
f1 = []
for s in thresholds:
    sig = p_exact >= s
    tp, fp, fn = (sig & is_spam).sum(), (sig & ~is_spam).sum(), (~sig & is_spam).sum()
    f1.append(2 * tp / (2 * tp + fp + fn))
f1 = np.array(f1)
best = thresholds[np.isclose(f1, f1.max())]
print(f'maximum F1 {f1.max():.2f}, for thresholds from {best.min():.2f} to {best.max():.2f}')
print('M6: exact value', round(float(p_exact[5]), 4), '→ flagged at threshold 0.27?', bool(p_exact[5] >= 0.27))

### c) to e) Annotated solution

**c)** The best threshold does rise with $k$: from 0.27–0.37 for $k = 0.1$ to 0.51–0.62 for $k = 3$ or 10. Beyond $k = 3$, it no longer moves: the only remaining false alarm, M3 (0.5), is already avoided, and going higher would only miss spams.

**d)** Between two neighboring estimated probabilities, no message changes category: the confusion matrix, and therefore the cost, is the same for all thresholds in the interval. For $k = 1$, the thresholds from 0.27 to 0.37 (one false alarm, none missed) and from 0.51 to 0.62 (no false alarm, one missed) all cost 1; in between, from 0.38 to 0.50, M3 and M7 are on the wrong side and the cost is 2. M6 is not flagged at threshold 0.27 because its exact value is 0.2689…: its “0.27” is only a rounded value.

**e)** The F1 is maximal (0.89) for the thresholds from 0.27 to 0.37. The cost points to the same thresholds when false alarms are cheap ($k = 0.1$), and includes them in a tie for $k = 1$. As soon as $k$ reaches 3, cost and F1 part ways: this is the association's situation (lesson 5.5).

## Exercise 5.5 — Predict, then check: rounding at the edge of the threshold

*Concepts: decision rule (equation 5.1), sigmoid (equation 5.4), F1 (equation 5.8).* Same weights, but a constant b = −4.01.

### a) to c) and e) Annotated solution

**a)** A common prediction: “nothing changes, a drop of 0.01 is tiny.” Rounded to two decimals, the twelve probabilities are indeed the same as in the table of lesson 5.2. Yet M3 changes side.

**b)** $z = 1 \times 4 + 0.5 \times 0 - 4.01 = -0.01$. Then $e^{0.01} \approx 1.01005$ and $\hat p = 1 / 2.01005 \approx 0.4975$. Rounded to two decimals: 0.50. But $0.4975 < 0.5$: M3 **is not flagged**. Reading 0.50 in a rounded table would lead to the wrong decision.

**c)** At threshold 0.5: $TP = 3$, $FN = 1$ (M7), $FP = 0$, $TN = 8$. Precision $3/3 = 1$, recall $3/4 = 0.75$, $F_1 = 6/7 \approx 0.86$, as at threshold 0.6 in the table of lesson 5.5: the false alarm M3 has disappeared. At threshold 0.25, nothing changes: M6 is now 0.2670…, still above 0.25.

**e)** The rule compares the value computed by the filter with the threshold, not the way it is displayed. Rounding can push a value across the threshold in either direction. When rounded probabilities are published, one must therefore say whether the decisions were made on the exact values, and point out the cases at the edge of the threshold.

In [ ]:
p_m3 = float(sigmoid(1 * 4 + 0.5 * 0 - 4.01))   # z = −0.01 → 0.4975…
flagged_m3 = p_m3 >= 0.5                          # False: the exact value is below the threshold

In [ ]:
# Check, then d) decisions on the exact values and on the rounded values
if p_m3 is None or flagged_m3 is None:
    print('not yet completed. Complete the previous cell.')
else:
    print(f'M3: p̂ = {p_m3:.4f} (displayed {p_m3:.2f}), flagged: {flagged_m3} →',
          'correct' if abs(p_m3 - 0.4975) < 0.0001 and flagged_m3 is False else 'check again')
p_coll = sigmoid(X @ w - 4.01)
for threshold in (0.5, 0.25):
    exact, rounded = p_coll >= threshold, p_coll.round(2) >= threshold
    diff = list(twelve['message'][exact != rounded])
    print(f'threshold {threshold}: different decisions between exact and rounded value: {diff if diff else "none"}')
sig = p_coll >= 0.5
tp, fp, fn = int((sig & is_spam).sum()), int((sig & ~is_spam).sum()), int((~sig & is_spam).sum())
print(f'colleague, threshold 0.5: TP = {tp}, FN = {fn}, FP = {fp}; precision = {tp / (tp + fp):.2f}, '
      f'recall = {tp / (tp + fn):.2f}, F1 = {2 * tp / (2 * tp + fp + fn):.2f}')

## Exercise 5.6 — Challenge: going down the log loss

*Concepts: mean loss (equation 5.7), partial derivatives and gradient (equation 3.6, lesson 3.4), sigmoid (equation 5.4).*

### a) to c) Annotated solution

**a)** For a spam ($y = 1$), the loss is $-\ln(\sigma(z))$. Its derivative is $-\sigma'(z) / \sigma(z) = -(1 - \sigma(z)) = \hat p - 1 = \hat p - y$. For a legitimate message ($y = 0$), the loss is $-\ln(1 - \sigma(z))$, with derivative $\sigma'(z) / (1 - \sigma(z)) = \sigma(z) = \hat p - 0 = \hat p - y$. In both cases: $\hat p - y$.

**b)** Since $z_i = w_1 x_{i1} + w_2 x_{i2} + b$, the derivative of $z_i$ with respect to $b$ is 1 and with respect to $w_j$ it is $x_{ij}$. Averaging over the messages: $\partial L / \partial b = \frac{1}{n} \sum (\hat p_i - y_i)$ and $\partial L / \partial w_j = \frac{1}{n} \sum (\hat p_i - y_i)\, x_{ij}$. This is the same form as for the MSE (equation 3.6): a **gap between prediction and reality**, multiplied by the feature. $\hat p_i - y_i$ plays the role of the gap of chapter 3, up to the sign and the factor 2.

**c)** With the rounded probabilities of lesson 5.2: $\sum \hat p_i = 4.08$ and there are 4 spams, so $\partial L / \partial b = (4.08 - 4) / 12 \approx 0.007$ (0.006 with the exact values). The positive sign says that $b$ should be lowered a little; the size, almost zero, says that $b = -4$ is already almost at the bottom of the loss curve of lesson 5.4, whose minimum is near $-4.05$. At the bottom, the sum of the estimated probabilities equals the number of spams.

In [ ]:
def gradient(w_, b_):
    """Returns (dL/dw, dL/db): dL/dw is an array of two numbers."""
    gap = sigmoid(X @ w_ + b_) - y          # p̂ − y: the “gap” of each message
    return X.T @ gap / len(y), gap.mean()  # mean of gap × feature, mean of gap

In [ ]:
# Check, then d) 1,000 gradient steps (η = 0.5) from w = (1, 0.5), b = −4
def avg_loss(w_, b_):
    p_ = sigmoid(X @ w_ + b_)
    return float(np.mean(-(y * np.log(p_) + (1 - y) * np.log(1 - p_))))

if gradient(w, b) is None:
    print('not yet completed. Complete the function.')
else:
    gw, gb = gradient(w, b)
    print(f'at the start: dL/db = {gb:.4f}, dL/dw = {np.round(gw, 4)} →',
          'correct' if abs(gb - 0.0062) < 0.0001 and np.allclose(gw, [0.0144, -0.2368], atol=0.0001) else 'check again')
    w_, b_ = w.astype(float), float(b)
    for step_k in range(1, 1001):
        gw, gb = gradient(w_, b_)
        w_, b_ = w_ - 0.5 * gw, b_ - 0.5 * gb
        if step_k in (1, 10, 100, 1000):
            print(f'after {step_k:>4} steps: loss = {avg_loss(w_, b_):.4f}, w = {np.round(w_, 2)}, b = {b_:.2f}')
    print('final loss →', 'correct' if abs(avg_loss(w_, b_) - 0.0308) < 0.0005 else 'check again')
# e) Weights that perfectly separate the twelve messages: suspicious words ≥ 3 for spams, ≤ 2 for legitimate ones
for factor in (1, 10):
    print(f'w = (0, {2 * factor}), b = {-5 * factor}: mean loss = {avg_loss(np.array([0, 2.0]) * factor, -5.0 * factor):.6f}')

### d) and e) Annotated solution

**d)** The mean loss goes down from 0.26 to about 0.031 after 1,000 steps, and it would keep going down. The weights, however, change a lot: the weight of the suspicious words rises toward 4.4, the constant goes down toward −9.2, and the weight of the links becomes **negative** (about −0.54). The model has noticed that M3, the newsletter, has many links and is not a spam. On twelve messages, this is a coincidence, not a rule: nothing says that a link makes a message less suspicious.

**e)** With $w = (0, 2)$ and $b = -5$, all spams have $z \geq 1$ and all legitimate messages $z \leq -1$: the mean loss is about 0.10. With weights ten times larger, it drops to about 0.00001. It can get as close to 0 as we like, without ever reaching it: the twelve messages are **separable**. But a filter this sure of itself would be disastrous on a spam next week that had only two suspicious words: its loss would be huge. This is why scikit-learn adds regularization, which slows down the growth of the weights. And this is the question of chapter 6: a loss measured on the messages used for fitting does not say what will happen on the others.